# Phase 3: Exploratory Data Analysis

This notebook documents Phase 3 for the Bosch Production Line Performance project.

Phase 3 objectives:

13. Calculate failure rate by production line.
14. Calculate failure rate by station.
15. Identify high-risk stations.
16. Visualize product flow paths.
17. Analyze time-based patterns and seasonality.
18. Create correlation and distribution reports.

The analysis uses the raw CSV datasets directly. It builds station presence, line presence, product flow paths, and time features from `train_date.csv`, while joining `Response` from `train_numeric.csv`.

## 1. Import Libraries and Phase 3 Helpers

In [ ]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT))

from src.data.phase3_exploratory_data_analysis import (
    REPORTS_DIR,
    FIGURES_DIR,
    build_train_flow_from_raw,
    analyze_categorical_one_hot_patterns,
    calculate_line_failure_rates,
    calculate_station_failure_rates,
    create_correlation_report,
    create_distribution_report,
    create_flow_path_summary,
    identify_high_risk_stations,
    analyze_time_patterns,
    save_plots,
    write_report,
)

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 180)

print('Project root:', PROJECT_ROOT)

## 2. Build Raw-Sourced Manufacturing Flow Features

For this phase, the station and line features are rebuilt directly from the raw CSVs. `train_date.csv` gives station presence and relative time features, while `train_numeric.csv` supplies the `Response` target. No Phase 2 Parquet flow dataset is used.

In [ ]:
train_flow, time_features = build_train_flow_from_raw(chunksize=20_000)

print(train_flow.shape)
train_flow.head()

## 3. Failure Rate by Production Line

A row can touch more than one line, so each line failure rate is calculated among parts where that line is present.

In [ ]:
line_rates = calculate_line_failure_rates(train_flow)
line_rates

## 4. Failure Rate by Station

Station-level risk is calculated the same way: among parts where a station is present, what share failed?

In [ ]:
station_rates = calculate_station_failure_rates(train_flow)
station_rates.head(15)

## 5. Identify High-Risk Stations

A high-risk station is one with enough volume and a failure rate above the overall training failure rate. The risk score combines excess failure rate with station volume, so tiny stations do not dominate the ranking.

In [ ]:
high_risk = identify_high_risk_stations(station_rates, min_parts=1_000)
high_risk.head(20)

## 6. Product Flow Paths

The flow path summary groups rows by their station-presence pattern. This shows the most common manufacturing routes and the failure rate for each route.

In [ ]:
flow_paths = create_flow_path_summary(train_flow, top_n=50)
flow_paths.head(15)

## 7. Time-Based Patterns and Seasonality

Bosch date values are anonymized relative production times, not calendar timestamps. Because of that, this notebook analyzes time patterns using relative-time bins rather than calendar seasonality.

The raw-sourced builder already scanned `train_date.csv` in chunks and created compact row-level time features:

- `first_event_time`
- `last_event_time`
- `process_duration`
- `observed_date_values`

In [ ]:
time_features.head()

In [ ]:
first_time_patterns, duration_patterns = analyze_time_patterns(time_features, bins=20)

display(first_time_patterns)
display(duration_patterns)

## 8. Flow Correlation Report

This report measures correlation between engineered flow features and `Response`. These are not causal claims; they are a screening view for features that may be useful in modeling.

In [ ]:
correlations = create_correlation_report(train_flow)
correlations.head(30)

## 9. Raw Categorical One-Hot EDA

`train_categorical.csv` has thousands of sparse categorical columns, so full one-hot encoding would create an unnecessarily large EDA matrix. Instead, this step selects the most complete categorical columns, one-hot encodes those selected columns in chunks, and aggregates category-level failure rates and correlations with `Response`.


In [ ]:
priority_station_keys = high_risk['station_key'].head(10).tolist()
categorical_selected_columns, categorical_encoded_summary, categorical_correlations = analyze_categorical_one_hot_patterns(
    max_columns=30,
    chunksize=20_000,
    priority_station_keys=priority_station_keys,
)

display(categorical_selected_columns)
display(categorical_encoded_summary.head(30))
display(categorical_correlations.head(30))

## 10. Distribution Report

The distribution report compares flow and time features between non-failed (`Response = 0`) and failed (`Response = 1`) parts.

In [ ]:
distributions = create_distribution_report(train_flow, time_features)
distributions

## 11. Save Reports and Visualizations

The final cell saves CSV reports, PNG figures, and the markdown summary for Phase 3.

In [ ]:
line_rates.to_csv(REPORTS_DIR / 'phase3_line_failure_rates.csv', index=False)
station_rates.to_csv(REPORTS_DIR / 'phase3_station_failure_rates.csv', index=False)
high_risk.to_csv(REPORTS_DIR / 'phase3_high_risk_stations.csv', index=False)
flow_paths.to_csv(REPORTS_DIR / 'phase3_flow_path_summary.csv', index=False)
first_time_patterns.to_csv(REPORTS_DIR / 'phase3_first_time_failure_patterns.csv', index=False)
duration_patterns.to_csv(REPORTS_DIR / 'phase3_duration_failure_patterns.csv', index=False)
correlations.to_csv(REPORTS_DIR / 'phase3_correlation_report.csv', index=False)
distributions.to_csv(REPORTS_DIR / 'phase3_distribution_report.csv', index=False)
categorical_selected_columns.to_csv(REPORTS_DIR / 'phase3_categorical_ohe_selected_columns.csv', index=False)
categorical_encoded_summary.to_csv(REPORTS_DIR / 'phase3_categorical_ohe_failure_rates.csv', index=False)
categorical_correlations.to_csv(REPORTS_DIR / 'phase3_categorical_ohe_correlation_report.csv', index=False)

figures = save_plots(
    line_rates=line_rates,
    station_rates=station_rates,
    high_risk=high_risk,
    flow_paths=flow_paths,
    first_time_patterns=first_time_patterns,
    duration_patterns=duration_patterns,
    correlations=correlations,
    categorical_encoded_summary=categorical_encoded_summary,
    train_flow=train_flow,
)

write_report(
    line_rates=line_rates,
    station_rates=station_rates,
    high_risk=high_risk,
    flow_paths=flow_paths,
    first_time_patterns=first_time_patterns,
    duration_patterns=duration_patterns,
    correlations=correlations,
    distributions=distributions,
    categorical_selected_columns=categorical_selected_columns,
    categorical_encoded_summary=categorical_encoded_summary,
    categorical_correlations=categorical_correlations,
    figures=figures,
)

print('Saved Phase 3 report:', REPORTS_DIR / 'phase3_exploratory_data_analysis_report.md')
figures

## 12. Phase 3 Deliverables

After running this notebook, the project has:

- Failure rate by production line.
- Failure rate by station.
- High-risk station ranking.
- Product flow path summary.
- Relative-time failure pattern summaries.
- Correlation report.
- Distribution report.
- Visual charts in `reports/figures/`.